# Solution F2 · KNN, logistic and random forest on four shapes of data

**Type**: Compare

## Objective

Compare three models (scaled KNN with `k=15`, logistic regression, random forest of 100 trees) on **four datasets of different shapes**: nonlinear boundary in 2D, linear boundary in 40 dimensions, nonlinear boundary with 30 noise columns, and a
large dataset. Measure accuracy, training time and prediction time, and summarize when each model is the right choice.

In [ ]:
import sys, pathlib
_root = next(p for p in [pathlib.Path.cwd(), *pathlib.Path.cwd().parents] if (p / "common").is_dir())
sys.path.insert(0, str(_root))

import warnings; warnings.filterwarnings("ignore")     # clean output: here the warnings are just noise
import numpy as np, pandas as pd
import matplotlib.pyplot as plt
from common import datasets as ds
from common.verify import check, close, done, verify

import time
from sklearn.datasets import make_classification, make_moons
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.neighbors import KNeighborsClassifier
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler


def datasets():
    rng = np.random.default_rng(0)
    moons, y_moons = make_moons(3000, noise=0.3, random_state=0)
    Xl = rng.normal(size=(3000, 40))
    w = np.r_[rng.normal(size=10), np.zeros(30)]                     # only 10 features out of 40 matter, linearly
    yl = rng.binomial(1, 1 / (1 + np.exp(-3 * Xl @ w / np.linalg.norm(w))))
    large = make_classification(40000, n_features=20, n_informative=8, n_redundant=0, class_sep=1.0, random_state=0)
    return {"moons 2D": (moons, y_moons), "linear 40D": (Xl, yl), "moons + 30 noise": (np.c_[moons, rng.normal(size=(3000, 30))], y_moons), "large": large}


models = {
    "knn": lambda: make_pipeline(StandardScaler(), KNeighborsClassifier(15)),
    "logistic": lambda: make_pipeline(StandardScaler(), LogisticRegression(max_iter=1000)),
    "forest": lambda: RandomForestClassifier(100, random_state=0, n_jobs=-1),
}

## Commented solution

In [ ]:
rows = {}
for name, (X, y) in datasets().items():
    X_tr, X_te, y_tr, y_te = train_test_split(X, y, test_size=0.3, random_state=0)
    for model_name, make in models.items():
        m = make()
        t0 = time.time(); m.fit(X_tr, y_tr); fit_time = time.time() - t0
        t0 = time.time(); acc = m.score(X_te, y_te); pred_time = time.time() - t0
        rows[(name, model_name)] = {"acc": acc, "fit_time": fit_time, "pred_time": pred_time}
table = pd.DataFrame(rows).T
table.round(3)

In [ ]:
a = table["acc"].unstack()
best, worst = a.idxmax(axis=1).to_dict(), a.idxmin(axis=1).to_dict()
knn_fastest_fit = bool(table.loc["large", "fit_time"].idxmin() == "knn")
pred_ratio = table.loc[("large", "knn"), "pred_time"] / table.loc[("large", "logistic"), "pred_time"]
for_low_latency = "logistic"
for_noise = "forest"
print("best:", best, "\nworst:", worst, f"\nKNN on the large dataset: fastest fit = {knn_fastest_fit}, prediction {pred_ratio:.0f} times slower than the logistic")

## Check

In [ ]:
with verify():
    check(table.shape == (12, 3) and list(table.columns) == ["acc", "fit_time", "pred_time"], "12 rows (4 datasets x 3 models) and three columns", "acc, fit_time, pred_time")
    check(best["linear 40D"] == "logistic" and worst["linear 40D"] == "knn", "on the linear boundary in 40 dimensions the logistic wins and the KNN loses", "compare the accuracies")
    check(worst["moons + 30 noise"] == "knn" and best["moons + 30 noise"] == "forest", "with 30 noise columns the forest wins and the KNN is last", "compare the accuracies")
    check(table.loc[("moons 2D", "logistic"), "acc"] < table.loc[("moons 2D", "knn"), "acc"] - 0.03 and worst["moons 2D"] == "logistic", "on the 2D moons the logistic (linear boundary) loses to the KNN by over 3 points", "the boundary is not linear")
    check(knn_fastest_fit is True and pred_ratio > 20, "on the large dataset the KNN has the fastest fit and prediction over 20 times slower than the logistic", "KNN predict computes the distances from the whole training set")
    check(for_low_latency == "logistic" and for_noise == "forest", "low latency: logistic; robustness to noise: forest", "summary of the comparison")
done()

## What you learned

- **No model wins everywhere.** On the moons (nonlinear boundary in 2 dimensions) the KNN is first (0.91) and the logistic last (0.85: a linear boundary cannot follow a curve). On the **linear** boundary in 40 dimensions the logistic wins (0.82) and the KNN is last (0.74, distances in
  many dimensions with 30 useless columns). With 30 noise columns added to the moons the forest stays at 0.89 while the KNN drops to 0.80 (below the logistic, 0.85). The forest is never last: it is the model most robust to different shapes, at the price of training time and a larger model.
- **Different costs.** The KNN has almost no fit (0.02 s on the 28,000 examples) but pays at prediction: on the large dataset it is about 140 times slower than the logistic (1 s against 7 ms for 12,000 points). The forest has the most expensive fit (2.4 s against 0.08 s for the logistic) but a
  fast prediction. The logistic has both minimal. To serve thousands of requests per second on a large dataset, the "pure" KNN is out of the question without an approximate index.
- The comparison has value because it is made **on the same data and with the same split**; but it is still a single split: differences of one or two points (2D moons: KNN 0.91 against forest 0.90; linear: logistic 0.82 against forest 0.81) are not significant. To decide you need repeated
  cross-validation (F1 of module 3).
- A procedure follows: look at the shape of the data (dimension, linearity, noise, number of rows), **always** try a linear baseline and a tree one, and use KNN if and only if the proximity structure is the heart of the problem.

## Common pitfall

Concluding from a comparison on a single dataset (the moons, which are the case KNN likes best) that "KNN is better than the logistic", or from the large dataset that "KNN is unusable". The answer depends on the shape of the data, and the comparison must be repeated on yours.

## Interview question

<details><summary>Tabular data, 50 features, 200,000 rows, classification goal: where do you start?</summary>

From a linear baseline (regularized logistic) and a tree model (random forest or gradient boosting): they are robust, scale well and at dimension 50 the KNN would have problems both of distance and of prediction cost. I compare with cross-validation, look at the calibration and
the required latency. I reserve KNN for problems where proximity has a direct meaning (recommendations, similarity search, geographic data) and in that case with approximate indexes.

</details>